# 커밋과 .gitignore

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.


## 준비

임시 저장소를 만들고 이 노트북에만 적용되는 작성자 정보를 설정합니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 저장소를 main 브랜치로 초기화합니다.
repo = root / "orders"
repo.mkdir()
git("init", "-b", "main", cwd=repo)
configure(repo)
print(repo.name, git("branch", "--show-current", cwd=repo))

orders main


## 기본 실행

무시 규칙을 첫 커밋에, 프로그램을 다음 커밋에 기록합니다.

In [3]:
# 무시 규칙과 기능 코드를 나누어 기록하고 두 커밋의 메시지를 확인합니다.
(repo / "app.py").write_text('print("주문 2건")\n', encoding="utf-8")
(repo / ".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")
(repo / ".env").write_text("APP_MODE=demo\n", encoding="utf-8")
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 생성 파일 추적 제외", cwd=repo)
git("add", "app.py", cwd=repo)
git("commit", "-m", "feat: 주문 요약 추가", cwd=repo)
print(git("status", "--short", "--ignored", cwd=repo))
print(git("log", "-2", "--pretty=%s", cwd=repo))

!! .env
feat: 주문 요약 추가
chore: 생성 파일 추적 제외


## 값 하나를 바꿔 비교

출력 숫자만 2에서 3으로 바꾸고 차이를 확인합니다.

In [4]:
# 파일의 출력 숫자 하나를 바꾼 뒤 diff로 실제 변경을 읽습니다.
(repo / "app.py").write_text('print("주문 3건")\n', encoding="utf-8")
print(git("diff", "--", "app.py", cwd=repo))
git("add", "app.py", cwd=repo)
git("commit", "-m", "fix: 주문 건수 수정", cwd=repo)
print(git("log", "--oneline", "-2", cwd=repo))

diff --git a/app.py b/app.py
index 0bde456..4333761 100644
--- a/app.py
+++ b/app.py
@@ -1 +1 @@
-print("주문 2건")
+print("주문 3건")
ac03736 fix: 주문 건수 수정
428557e feat: 주문 요약 추가


## 실패 재현

이미 추적 중인 파일은 나중에 무시 규칙을 추가해도 추적 상태가 유지됩니다.

In [5]:
# 이미 추적하는 debug.log는 무시 규칙을 추가해도 변경 대상으로 남습니다.
tracked = repo / "debug.log"
tracked.write_text("first run\n", encoding="utf-8")
git("add", "debug.log", cwd=repo)
git("commit", "-m", "chore: 디버그 로그 기록", cwd=repo)
with (repo / ".gitignore").open("a", encoding="utf-8") as file:
    file.write("*.log\n")
tracked.write_text("second run\n", encoding="utf-8")
print("추적 여부:", git("ls-files", "debug.log", cwd=repo))
print("변경 상태:", git("status", "--short", cwd=repo))

추적 여부: debug.log
변경 상태: M .gitignore
 M debug.log


## 수정

색인에서만 파일을 제거하면 작업 폴더의 파일은 남고 이후 변경은 무시됩니다.

In [6]:
# --cached는 색인에서만 제거하므로 작업 폴더의 로그 파일은 유지됩니다.
git("rm", "--cached", "debug.log", cwd=repo)
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 로그 파일 추적 제외", cwd=repo)
tracked.write_text("third run\n", encoding="utf-8")
print("파일 존재:", tracked.exists())
print("추적 결과:", git("ls-files", "debug.log", cwd=repo) or "없음")
print("무시 규칙:", git("check-ignore", "-v", "debug.log", cwd=repo))
assert tracked.exists() and not git("ls-files", "debug.log", cwd=repo)

파일 존재: True
추적 결과: 없음
무시 규칙: .gitignore:3:*.log	debug.log
